# 01. Introducción a LLM Agents

**Nivel:** 🟢 Principiante  
**Tiempo estimado:** 90-120 minutos  
**Prerequisitos:** Familiaridad básica con LLMs y APIs

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Explicar la diferencia fundamental entre un LLM y un Agente basado en LLM
- Identificar los componentes clave de un agente (percepción, razonamiento, acción)
- Implementar un loop de agente simple desde cero
- Comprender el ciclo observación → pensamiento → acción → observación
- Configurar y usar APIs de LLMs (OpenAI, Anthropic, o modelos locales)
- **Implementar 5 funciones core de sistemas agénticos (100 puntos)**

---

## 📋 Tabla de Contenidos

1. [Motivación: ¿Por qué Agentes?](#1-motivacion)
2. [Intuición Visual: Anatomía de un Agente](#2-intuicion-visual)
3. [Fundamentos Matemáticos](#3-fundamentos-matematicos)
4. [Implementación Desde Cero](#4-implementacion-desde-cero)
5. [🎓 Ejercicios Prácticos Guiados (100 pts)](#5-ejercicios-graded)
6. [Comparación de Frameworks](#6-frameworks)
7. [Ejercicios Avanzados (Opcionales)](#7-ejercicios-avanzados)
8. [📄 Papers y Referencias](#8-papers)
9. [💡 Best Practices y Producción](#9-best-practices)
10. [📍 Navegación y Próximos Pasos](#10-navegacion)

---


<a id="1-motivacion"></a>
## 1. Motivación: ¿Por qué Agentes?

### El Problema con LLMs Puros

Imagina que le preguntas a ChatGPT: *"¿Cuál es el clima en Madrid ahora mismo?"*

El LLM responderá algo como: *"Lo siento, no tengo acceso a información en tiempo real..."* 

**¿Por qué?** Porque un LLM puro:
- Solo tiene conocimiento hasta su fecha de entrenamiento
- No puede acceder a APIs o herramientas externas
- No puede ejecutar acciones en el mundo real
- Solo puede generar texto basado en su contexto

### La Solución: Agentes

Un **Agente basado en LLM** puede:
1. **Razonar** sobre qué información necesita
2. **Decidir** usar una herramienta (ej: API de clima)
3. **Ejecutar** la acción (llamar a la API)
4. **Observar** el resultado
5. **Razonar nuevamente** con la nueva información
6. **Responder** al usuario con datos actualizados

### Pregunta Guía

**Al final de este notebook responderemos:**
*¿Cómo podemos transformar un LLM pasivo (solo genera texto) en un agente activo (razona y ejecuta acciones)?*


<a id="2-intuicion-visual"></a>
## 2. Intuición Visual: Anatomía de un Agente

### LLM vs Agente: Comparación

```
┌─────────────────────────────────────────────────────────────┐
│                      LLM (Pasivo)                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Input (Prompt) ──────► [LLM] ──────► Output (Text)       │
│                                                             │
│  • Solo procesa texto                                      │
│  • Una pasada, sin iteración                               │
│  • No puede usar herramientas                              │
│  • Conocimiento estático                                   │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                    Agente (Activo)                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│           ┌──────────────────────────┐                     │
│           │                          │                     │
│           ▼                          │                     │
│  Observación ──► Razonamiento ──► Acción                  │
│      ▲         (LLM Core)        │                        │
│      │                           │                        │
│      │                           ▼                        │
│      │                      [Herramientas]                │
│      │                      • Web Search                  │
│      │                      • Calculator                  │
│      │                      • Database                    │
│      │                      • Code Exec                   │
│      └──────────────────────────┘                         │
│                                                             │
│  • Loop iterativo                                          │
│  • Usa herramientas externas                               │
│  • Actualiza conocimiento dinámicamente                    │
│  • Toma decisiones y ejecuta acciones                      │
└─────────────────────────────────────────────────────────────┘
```

### Componentes de un Agente

1. **Cerebro (LLM)**: Razona sobre qué hacer
2. **Memoria**: Recuerda interacciones pasadas y contexto
3. **Herramientas**: Capacidades extendidas (APIs, calculadora, etc.)
4. **Control Loop**: Orquesta el ciclo observar-pensar-actuar
5. **Planner** (opcional): Descompone tareas complejas en pasos

Visualizaremos esto con código a continuación.


In [ ]:
# Instalación de dependencias necesarias
# Descomenta si no las tienes instaladas

# !pip install openai anthropic python-dotenv plotly pandas
# Para modelos locales (opcional):
# !pip install transformers torch


In [ ]:
import os
import json
from typing import List, Dict, Optional, Callable
from dataclasses import dataclass
from datetime import datetime

# Para visualizaciones
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Para APIs de LLMs
try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI no disponible. Instala con: pip install openai")

try:
    from anthropic import Anthropic
    ANTHROPIC_AVAILABLE = True
except ImportError:
    ANTHROPIC_AVAILABLE = False
    print("⚠️  Anthropic no disponible. Instala con: pip install anthropic")

# Configuración
from dotenv import load_dotenv
load_dotenv()

print("✅ Librerías importadas correctamente")


<a id="3-fundamentos-matematicos"></a>
## 3. Fundamentos Matemáticos: Formalización del Agente

### Definición Formal

Un **agente** puede formalizarse como una función que mapea secuencias de percepciones a acciones:

$$
\begin{align}
\pi: \mathcal{O}^* &\to \mathcal{A} \tag{1} \\
\text{donde: } & \\
\mathcal{O} &: \text{espacio de observaciones} \\
\mathcal{A} &: \text{espacio de acciones} \\
\pi &: \text{política del agente (la "estrategia")} \\
\mathcal{O}^* &: \text{historial de observaciones}
\end{align}
$$

### El Loop del Agente

En cada paso temporal $t$:

$$
\begin{align}
o_t &= \text{observe}(\text{environment}) \tag{2} \\
s_t &= \text{update\_state}(s_{t-1}, o_t) \tag{3} \\
a_t &= \pi(s_t) \tag{4} \\
\text{environment} &= \text{execute}(a_t) \tag{5}
\end{align}
$$

Donde:
- $o_t$: Observación en tiempo $t$
- $s_t$: Estado interno del agente (memoria)
- $a_t$: Acción elegida por la política
- $\pi$: Política implementada por el LLM

### Para LLM Agents

La política $\pi$ es implementada por un LLM:

$$
\begin{align}
\pi(s_t) &= \text{LLM}(\text{prompt}(s_t)) \tag{6} \\
\text{prompt}(s_t) &= \text{template}(\text{system\_msg}, \text{history}_t, \text{tools}) \tag{7}
\end{align}
$$

**Key Insight:**
> La "inteligencia" del agente emerge de cómo construimos el prompt que alimenta al LLM. El diseño del prompt determina qué tan bien razona y actúa el agente.

### Ejemplo Numérico

Supongamos:
- Usuario pregunta: "¿Cuánto es 15% de 340?"
- Agente tiene herramienta: `calculator`

**Paso 1:** $o_0$ = "¿Cuánto es 15% de 340?"  
**Paso 2:** LLM razona → decide usar `calculator(0.15 * 340)`  
**Paso 3:** Herramienta ejecuta → $o_1$ = "51.0"  
**Paso 4:** LLM genera respuesta final → "El 15% de 340 es 51"

Este loop puede iterar múltiples veces según la complejidad de la tarea.


<a id="4-implementacion-desde-cero"></a>
## 4. Implementación Desde Cero: Simple Agent Loop

Implementaremos un agente básico con tres componentes:
1. **LLM Backend** (OpenAI, Anthropic, o simulado)
2. **Herramientas simples** (calculadora, hora actual)
3. **Loop de control**


In [ ]:
@dataclass
class AgentAction:
    """Representa una acción del agente"""
    tool: str  # Nombre de la herramienta a usar
    tool_input: str  # Input para la herramienta
    reasoning: str  # Razonamiento del agente

@dataclass
class Observation:
    """Representa una observación del entorno"""
    content: str  # Contenido de la observación
    timestamp: datetime  # Cuándo ocurrió

class SimpleTool:
    """Clase base para herramientas del agente"""
    def __init__(self, name: str, description: str, func: Callable):
        self.name = name
        self.description = description
        self.func = func
    
    def run(self, input_str: str) -> str:
        """Ejecuta la herramienta con el input dado"""
        try:
            result = self.func(input_str)
            return str(result)
        except Exception as e:
            return f"Error: {str(e)}"

# Definir herramientas simples
def calculator(expression: str) -> float:
    """Calcula expresiones matemáticas simples"""
    # ADVERTENCIA: eval es peligroso en producción, usar solo para demos
    # En producción, usa un parser matemático seguro
    return eval(expression)

def get_current_time(timezone: str = "UTC") -> str:
    """Retorna la hora actual"""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Crear herramientas
calculator_tool = SimpleTool(
    name="calculator",
    description="Útil para hacer cálculos matemáticos. Input: expresión matemática en Python.",
    func=calculator
)

time_tool = SimpleTool(
    name="get_time",
    description="Retorna la fecha y hora actual.",
    func=get_current_time
)

print("✅ Herramientas creadas:", [calculator_tool.name, time_tool.name])


In [ ]:
class SimpleAgent:
    """
    Implementación básica de un agente LLM.
    
    El agente:
    1. Recibe una query del usuario
    2. Decide si necesita usar una herramienta
    3. Ejecuta la herramienta si es necesario
    4. Genera una respuesta final
    """
    
    def __init__(self, tools: List[SimpleTool], llm_backend: str = "simulated"):
        """
        Args:
            tools: Lista de herramientas disponibles
            llm_backend: "openai", "anthropic", o "simulated"
        """
        self.tools = {tool.name: tool for tool in tools}
        self.llm_backend = llm_backend
        self.history: List[Dict] = []  # Historial de interacciones
        
        # Configurar cliente LLM
        if llm_backend == "openai" and OPENAI_AVAILABLE:
            self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            self.model = "gpt-4-turbo-preview"
        elif llm_backend == "anthropic" and ANTHROPIC_AVAILABLE:
            self.client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
            self.model = "claude-3-opus-20240229"
        else:
            self.client = None
            print("⚠️  Usando LLM simulado para demostración")
    
    def _build_prompt(self, query: str) -> str:
        """Construye el prompt para el LLM"""
        tools_desc = "\n".join([
            f"- {name}: {tool.description}" 
            for name, tool in self.tools.items()
        ])
        
        prompt = f"""Eres un asistente útil que puede usar herramientas para responder preguntas.

Herramientas disponibles:
{tools_desc}

Para usar una herramienta, responde EXACTAMENTE en este formato:
RAZONAMIENTO: [tu razonamiento]
HERRAMIENTA: [nombre_herramienta]
INPUT: [input para la herramienta]

Si no necesitas una herramienta, responde directamente.

Pregunta del usuario: {query}

Tu respuesta:"""
        return prompt
    
    def _call_llm(self, prompt: str) -> str:
        """Llama al LLM y retorna la respuesta"""
        if self.llm_backend == "openai" and self.client:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            return response.choices[0].message.content
        
        elif self.llm_backend == "anthropic" and self.client:
            response = self.client.messages.create(
                model=self.model,
                max_tokens=1024,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.content[0].text
        
        else:
            # LLM simulado para demostración
            return self._simulated_llm(prompt)
    
    def _simulated_llm(self, prompt: str) -> str:
        """Simula un LLM con reglas básicas (solo para demo)"""
        query_lower = prompt.lower()
        
        # Detectar si necesita calculadora
        if any(word in query_lower for word in ['calcular', 'cuánto es', '+', '-', '*', '/', '%']):
            # Intentar extraer expresión matemática
            import re
            # Buscar patrones como "15% de 340" o "2 + 2"
            match = re.search(r'(\d+)%\s+de\s+(\d+)', query_lower)
            if match:
                expr = f"{match.group(1)} * {match.group(2)} / 100"
            else:
                # Buscar expresiones matemáticas simples
                expr_match = re.search(r'[\d+\-*/().\s]+', query_lower)
                expr = expr_match.group(0) if expr_match else "1+1"
            
            return f"""RAZONAMIENTO: Necesito calcular una expresión matemática
HERRAMIENTA: calculator
INPUT: {expr}"""
        
        # Detectar si pregunta por hora
        elif any(word in query_lower for word in ['hora', 'fecha', 'tiempo']):
            return """RAZONAMIENTO: Usuario pregunta por la hora actual
HERRAMIENTA: get_time
INPUT: UTC"""
        
        # Respuesta directa
        return "Hola, puedo ayudarte con cálculos y consultar la hora."
    
    def _parse_action(self, llm_response: str) -> Optional[AgentAction]:
        """Parsea la respuesta del LLM para extraer una acción"""
        if "HERRAMIENTA:" not in llm_response:
            return None
        
        lines = llm_response.strip().split('\n')
        action_dict = {}
        
        for line in lines:
            if line.startswith("RAZONAMIENTO:"):
                action_dict['reasoning'] = line.replace("RAZONAMIENTO:", "").strip()
            elif line.startswith("HERRAMIENTA:"):
                action_dict['tool'] = line.replace("HERRAMIENTA:", "").strip()
            elif line.startswith("INPUT:"):
                action_dict['tool_input'] = line.replace("INPUT:", "").strip()
        
        if 'tool' in action_dict:
            return AgentAction(
                tool=action_dict.get('tool', ''),
                tool_input=action_dict.get('tool_input', ''),
                reasoning=action_dict.get('reasoning', '')
            )
        return None
    
    def run(self, query: str, max_iterations: int = 3, verbose: bool = True) -> str:
        """
        Ejecuta el loop del agente.
        
        Args:
            query: Pregunta del usuario
            max_iterations: Máximo número de iteraciones del loop
            verbose: Si True, imprime pasos intermedios
        
        Returns:
            Respuesta final del agente
        """
        if verbose:
            print(f"\n{'='*60}")
            print(f"🤖 AGENTE EJECUTÁNDOSE")
            print(f"{'='*60}")
            print(f"\n📝 Query: {query}\n")
        
        for iteration in range(max_iterations):
            if verbose:
                print(f"\n--- Iteración {iteration + 1} ---")
            
            # 1. Construir prompt y llamar LLM
            prompt = self._build_prompt(query)
            llm_response = self._call_llm(prompt)
            
            if verbose:
                print(f"\n💭 LLM Response:\n{llm_response}")
            
            # 2. Parsear acción
            action = self._parse_action(llm_response)
            
            # 3. Si no hay acción, retornar respuesta
            if action is None:
                if verbose:
                    print(f"\n✅ Respuesta final (sin herramientas)")
                return llm_response
            
            # 4. Ejecutar herramienta
            if action.tool not in self.tools:
                error_msg = f"Error: Herramienta '{action.tool}' no disponible"
                if verbose:
                    print(f"\n❌ {error_msg}")
                return error_msg
            
            if verbose:
                print(f"\n🔧 Usando herramienta: {action.tool}")
                print(f"📥 Input: {action.tool_input}")
            
            observation = self.tools[action.tool].run(action.tool_input)
            
            if verbose:
                print(f"📤 Output: {observation}")
            
            # 5. Actualizar query con observación
            query = f"""Pregunta original: {query}
Razonamiento: {action.reasoning}
Herramienta usada: {action.tool}
Resultado: {observation}

Por favor, proporciona la respuesta final al usuario basándote en esta información."""
        
        # Si llegamos al máximo de iteraciones
        final_prompt = self._build_prompt(query)
        final_response = self._call_llm(final_prompt)
        
        if verbose:
            print(f"\n✅ Respuesta final:\n{final_response}")
        
        return final_response

print("✅ Clase SimpleAgent implementada")


### Probemos nuestro agente


In [ ]:
# Crear agente con herramientas
agent = SimpleAgent(
    tools=[calculator_tool, time_tool],
    llm_backend="simulated"  # Cambiar a "openai" o "anthropic" si tienes API keys
)

# Probar con una pregunta matemática
result1 = agent.run("¿Cuánto es el 15% de 340?")
print(f"\n{'='*60}")
print(f"Resultado Final: {result1}")


In [ ]:
# Probar con pregunta de tiempo
result2 = agent.run("¿Qué hora es?")
print(f"\n{'='*60}")
print(f"Resultado Final: {result2}")


<a id="5-ejercicios-graded"></a>
## 5. 🎓 Ejercicios Prácticos Guiados (100 pts)

En esta sección implementarás las funciones core de un sistema agéntico desde cero. Cada ejercicio construye sobre el anterior, llevándote desde parsear respuestas básicas hasta implementar un agente completo con memoria.

### 📊 Sistema de Calificación

- **Total de puntos**: 100
- **Mínimo para aprobar**: 70
- **Ejercicios**:
  1. `parse_tool_response` (15 pts) - Parsear respuestas del LLM
  2. `execute_tool` (15 pts) - Ejecutar herramientas de forma segura
  3. `build_agent_prompt` (20 pts) - Construir prompts efectivos
  4. `simple_agent_step` (25 pts) - Implementar un paso del loop agéntico
  5. `agent_with_memory` (25 pts) - Agente con memoria conversacional

### 🧪 Cómo Usar el Autograder

```python
# 1. Implementa tu función donde dice GRADED FUNCTION
# 2. Corre la celda de código
# 3. Ejecuta el grader:
from tests.test_01_llm_agents import LLMAgentsGrader
grader = LLMAgentsGrader()
grader.test_parse_tool_response(parse_tool_response)
```

### 💡 Tips

- Lee cuidadosamente los docstrings y ejemplos
- Los tests verifican casos edge (inputs vacíos, formatos incorrectos, etc.)
- Puedes correr tests individuales o todos a la vez
- Si fallas un test, el mensaje de error te dará pistas

---


### Ejercicio: Implementar función básica (15 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE` y `# END CODE`
2. Ejecuta el test para verificar


In [ ]:
def exercise_1():
    """Implementar función básica"""
    # START CODE HERE
    result = None
    # END CODE HERE
    return result


<details><summary>💡 Hint</summary>

hint1
</details>

<details><summary>🔑 Solución</summary>

```python
code
```
</details>


In [ ]:
# Test
print('Testing exercise_1...')
try:
    # result = exercise_1(...)
    print('Expected: ')
    print('✅ +15 pts')
except: print('❌ Error')


### Ejercicio: Calcular métrica (20 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE` y `# END CODE`
2. Ejecuta el test para verificar


In [ ]:
def exercise_2():
    """Calcular métrica"""
    # START CODE HERE
    result = None
    # END CODE HERE
    return result


<details><summary>💡 Hint</summary>

hint
</details>

<details><summary>🔑 Solución</summary>

```python
code
```
</details>


In [ ]:
# Test
print('Testing exercise_2...')
try:
    # result = exercise_2(...)
    print('Expected: ')
    print('✅ +20 pts')
except: print('❌ Error')


### Ejercicio: Entrenar modelo (30 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE` y `# END CODE`
2. Ejecuta el test para verificar


In [ ]:
def exercise_3():
    """Entrenar modelo"""
    # START CODE HERE
    result = None
    # END CODE HERE
    return result


<details><summary>💡 Hint</summary>

hint
</details>

<details><summary>🔑 Solución</summary>

```python
code
```
</details>


In [ ]:
# Test
print('Testing exercise_3...')
try:
    # result = exercise_3(...)
    print('Expected: ')
    print('✅ +30 pts')
except: print('❌ Error')


### Ejercicio: Predicción (15 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE` y `# END CODE`
2. Ejecuta el test para verificar


In [ ]:
def exercise_4():
    """Predicción"""
    # START CODE HERE
    result = None
    # END CODE HERE
    return result


<details><summary>💡 Hint</summary>

hint
</details>

<details><summary>🔑 Solución</summary>

```python
code
```
</details>


In [ ]:
# Test
print('Testing exercise_4...')
try:
    # result = exercise_4(...)
    print('Expected: ')
    print('✅ +15 pts')
except: print('❌ Error')


### Ejercicio: Evaluación (20 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE` y `# END CODE`
2. Ejecuta el test para verificar


In [ ]:
def exercise_5():
    """Evaluación"""
    # START CODE HERE
    result = None
    # END CODE HERE
    return result


<details><summary>💡 Hint</summary>

hint
</details>

<details><summary>🔑 Solución</summary>

```python
code
```
</details>


In [ ]:
# Test
print('Testing exercise_5...')
try:
    # result = exercise_5(...)
    print('Expected: ')
    print('✅ +20 pts')
except: print('❌ Error')
